<a href="https://colab.research.google.com/github/Shamanth-V/DL-and-GenAI-Project/blob/main/dl-23f2004250-notebook-t22026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
import torch; print(torch.cuda.is_available())

True


# 🎯 Smart MCQ Solver Challenge
### Deep Learning & Generative AI Project : Diploma in BS Data Science and Applications

**Roll Number:** `23f2004250`
**Kaggle Competition:** Smart MCQ Solver Challenge · **Metric:** MAP@3

---

### 📌 Problem Statement
Each question in this dataset provides a **prompt** and **five candidate answers (A to E)**.
The task is to predict the **top-3 most likely correct answers**,
so that the true answer is retrieved as early as possible in the ranking. Performance is
scored with **Mean Average Precision @ 3 (MAP@3)**, which rewards the correct label
appearing higher (1st > 2nd > 3rd) in the predicted list.

### 🧭 Project Rubric Coverage

| # | Rubric Requirement | Model in this Notebook | Section |
|---|---|---|---|
| 1 | Classical / statistical baseline | **LightGBM** on TF-IDF similarity + engineered statistical features | Model 1 |
| 2 | **Model built from scratch** | **Feed-forward MLP** (pure PyTorch, no pretrained weights) trained on TF-IDF/SVD embeddings | Model 2 |
| 3 | **Pretrained model**, fine-tuned | **RoBERTa-base** fine-tuned end-to-end on `(prompt, option)` pairs | Model 3 |
| 4 | **Model of choice / bonus** | **LoRA-tuned RoBERTa** — parameter-efficient fine-tuning via `peft` | Model 4 |
| 5 | Ensembling milestone | Weighted probability average of the best 2–3 models by OOF MAP@3 | Ensemble |

Each model is evaluated with **GroupKFold cross-validation** (grouped by question `id`,
so no leakage of options from the same question across folds), and the **out-of-fold
(OOF) MAP@3** is reported for every model, this is the same set of comparable metrics
that will be logged to **Weights & Biases** for the milestone tracking requirement.

---

## 1️⃣ Environment Setup

In [48]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

### ⚙️ Dependency Fix : `peft` / `torchao` Version Conflict

`peft` (used later for **LoRA fine-tuning**, our Model 4 / bonus model) probes for an
optional `torchao` backend on import.

In [49]:
pip install peft

In [50]:
!pip uninstall -y torchao

In [51]:
!pip install -U peft

## WandB Tracking

`wandb` (Weights & Biases) is an experiment-tracking library, it logs
training runs (loss curves, metrics, hyperparameters, model checkpoints) so
we can compare runs on a dashboard instead of scrolling through console output.

### `import wandb`
Loads the wandb Python client into the notebook.

### `wandb.login(key="...")`
Authenticates the notebook session against your wandb account using an API key,
so any `wandb.init()` calls afterward know which account to log runs to.
ever appears in cell
output or gets saved into notebook version history.

In [52]:
import os
import wandb
wandb_key = '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'
wandb.login(key=wandb_key)
wandb.init(project="23f2004250-t22026", name="run 3")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


From-scratch MLP (OOF)/accuracy,▁
From-scratch MLP (OOF)/f1,▁
From-scratch MLP (OOF)/map3,▁
From-scratch MLP (OOF)/precision,▁
From-scratch MLP (OOF)/recall,▁
LightGBM (OOF)/accuracy,▁
LightGBM (OOF)/f1,▁
LightGBM (OOF)/map3,▁
LightGBM (OOF)/precision,▁
LightGBM (OOF)/recall,▁
+20,...


## 2️⃣ Pipeline Overview & Imports

The cell below sets up all shared configuration: random seed, the 5 answer-option
labels, cross-validation fold count, and environment-aware file paths (so the same
script works both on Kaggle and for local testing).

**Design principle:** every model in this notebook is evaluated using **GroupKFold**
cross-validation, grouped by question `id` — this guarantees that all 5 options
belonging to the same question always stay together in either the train or the
validation split, preventing information leakage between options of the same MCQ.

Smart MCQ Solver Challenge - Full Rubric Pipeline
Roll number: 23f2004250

Implements the required 5-part rubric:
  1. LightGBM        -> TF-IDF + statistical/engineered features   (classical baseline)
  2. From-scratch MLP -> trained on TF-IDF vectors, pure PyTorch, no pretrained weights
  3. Pretrained       -> fine-tuned RoBERTa on (prompt + option) pairs
  4. Bonus/unique     -> LoRA-tuned RoBERTa (parameter-efficient fine-tuning via PEFT)
  5. Ensemble          -> weighted average of the best 2-3 models probabilities -> top-3

In [53]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import TruncatedSVD

RANDOM_STATE = 42
OPTIONS = ["A", "B", "C", "D", "E"]
N_FOLDS = 2

ON_KAGGLE = os.path.exists("/kaggle/input")
OUT_DIR = "/kaggle/working"
#TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
#TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
#SUBMISSION_PATH = f"{OUT_DIR}/submission.csv"
TRAIN_PATH = "/content/train.csv"
TEST_PATH = "/content/test.csv"
SUBMISSION_PATH = "/content/submission.csv"
os.makedirs(OUT_DIR, exist_ok=True)

import warnings
warnings.filterwarnings("ignore")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

## 3️⃣ Data Loading, Reshaping & the MAP@3 Metric

**Key transformation : wide → long format:**
Each row in the raw dataset represents one *question* with 5 answer columns
(`A, B, C, D, E`). We reshape this into a **long format** where each row represents
one *(question, option)* pair with a binary label : `1` if that option is the correct
answer, `0` otherwise. This turns the ranking problem into a **binary classification
problem**: "is this specific option correct?", which every one of our 4 models solves,
each producing a probability per option that we then rank.

This section also implements:
- `word_set()` : tokenizes text into a set of lowercase words (used for lexical overlap features)
- `map_at_3()` : our exact competition metric, used for **all** OOF evaluation below
- `proba_to_ranked_letters()` : converts per-option probabilities into a ranked list of
  option letters (A to E) per question, ready for MAP@3 scoring or submission
- `evaluate()` : convenience wrapper that scores and prints a model's OOF MAP@3

In [54]:
def load_data():
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    return train, test


def to_long(df, is_train):
    rows = []
    for _, r in df.iterrows():
        for opt in OPTIONS:
            rows.append({
                "id": r["id"],
                "prompt": r["prompt"],
                "option_letter": opt,
                "option_text": r[opt],
                "label": int(is_train and r["answer"] == opt),
            })
    return pd.DataFrame(rows)


def word_set(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))


def map_at_3(y_true_letters, ranked_preds):
    scores = []
    for true, preds in zip(y_true_letters, ranked_preds):
        s = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                s = 1.0 / (i + 1)
                break
        scores.append(s)
    return float(np.mean(scores))


def proba_to_ranked_letters(long_df, proba):
    tmp = long_df.copy()
    tmp["proba"] = proba
    ranked = (
        tmp.sort_values(["id", "proba"], ascending=[True, False])
        .groupby("id")["option_letter"]
        .apply(list)
    )
    return ranked


from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

def evaluate(name, long_df, id_order, truth_series, proba, log_to_wandb=True, threshold=0.5):
    ranked = proba_to_ranked_letters(long_df, proba).reindex(id_order)
    score = map_at_3(truth_series.reindex(id_order).tolist(), ranked.tolist())
    print(f"[{name}] MAP@3 = {score:.4f}")

    y_true = long_df["label"].values
    y_pred = (proba >= threshold).astype(int)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    print(f"[{name}] Precision={precision:.4f} Recall={recall:.4f} F1={f1:.4f} Acc={accuracy:.4f}")

    if log_to_wandb:
        wandb.log({
            f"{name}/map3": score,
            f"{name}/precision": precision,
            f"{name}/recall": recall,
            f"{name}/f1": f1,
            f"{name}/accuracy": accuracy,
        })

    return score


## 4️⃣ Feature Engineering : TF-IDF, Statistical & Lexical Features

Before any modeling, we engineer features that describe each answer option **on its
own**, **relative to the question prompt**, and **relative to the other 4 options** in
the same question. Exploratory analysis on this dataset showed a strong, exploitable
signal: the correct answer is disproportionately the **longest** and most **detailed**
option among the five.

**Feature groups implemented:**
- **Length-based:** character/word counts, rank of this option's length among its
  siblings, z-score of length relative to the question's mean/std, "is longest" /
  "is shortest" flags
- **Positional prior:** one-hot encoding of the option letter (A to E), capturing any mild
  label imbalance in how correct answers are distributed
- **Prompt overlap:** word-level overlap between the option text and the question prompt
- **Inter-option overlap:** lexical similarity of an option to the *other* options in the
  same question (distractors often paraphrase each other; the correct answer can stand
  apart lexically)
- **Surface cues:** comma/digit counts, negation-word presence
- **TF-IDF cosine similarity** between option text and prompt (fit once, jointly across
  train + test to avoid vocabulary mismatch)
- **SVD-compressed dense embeddings** of the option text (used as numeric input features
  for the from-scratch MLP in Model 2)

In [55]:
def add_engineered_features(long_df):
    long_df = long_df.copy()
    long_df["opt_len_chars"] = long_df["option_text"].astype(str).str.len()
    long_df["opt_len_words"] = long_df["option_text"].astype(str).str.split().str.len()
    long_df["prompt_len_chars"] = long_df["prompt"].astype(str).str.len()
    long_df["prompt_len_words"] = long_df["prompt"].astype(str).str.split().str.len()

    grp_len = long_df.groupby("id")["opt_len_chars"]
    long_df["len_rank_pct"] = long_df.groupby("id")["opt_len_chars"].rank(pct=True)
    long_df["len_minus_mean"] = long_df["opt_len_chars"] - grp_len.transform("mean")
    long_df["len_zscore"] = long_df["len_minus_mean"] / (grp_len.transform("std") + 1e-6)
    long_df["is_longest"] = (long_df["opt_len_chars"] == grp_len.transform("max")).astype(int)
    long_df["is_shortest"] = (long_df["opt_len_chars"] == grp_len.transform("min")).astype(int)

    for opt in OPTIONS:
        long_df[f"is_opt_{opt}"] = (long_df["option_letter"] == opt).astype(int)

    prompt_words = long_df["prompt"].apply(word_set)
    option_words = long_df["option_text"].apply(word_set)
    overlap = [len(p & o) for p, o in zip(prompt_words, option_words)]
    long_df["prompt_overlap_count"] = overlap
    long_df["prompt_overlap_ratio"] = [c / (len(o) + 1e-6) for c, o in zip(overlap, option_words)]

    long_df["_wordset"] = option_words
    other_mean, other_max = [], []
    for qid, group in long_df.groupby("id"):
        sets = group["_wordset"].tolist()
        for i in range(len(sets)):
            sims = []
            for j in range(len(sets)):
                if i == j:
                    continue
                a, b = sets[i], sets[j]
                u = len(a | b)
                sims.append(len(a & b) / u if u else 0.0)
            other_mean.append(np.mean(sims) if sims else 0.0)
            other_max.append(np.max(sims) if sims else 0.0)
    long_df["other_overlap_mean"] = other_mean
    long_df["other_overlap_max"] = other_max
    long_df.drop(columns=["_wordset"], inplace=True)

    long_df["num_commas"] = long_df["option_text"].astype(str).str.count(",")
    long_df["num_digits"] = long_df["option_text"].astype(str).str.count(r"\d")
    long_df["has_negation"] = long_df["option_text"].astype(str).str.contains(
        r"\bnot\b|\bno\b|\bnever\b|\bcannot\b|\bdoes not\b|\bdon't\b", case=False
    ).astype(int)
    return long_df


ENGINEERED_FEATURES = [
    "opt_len_chars", "opt_len_words", "prompt_len_chars", "prompt_len_words",
    "len_rank_pct", "len_minus_mean", "len_zscore", "is_longest", "is_shortest",
    "is_opt_A", "is_opt_B", "is_opt_C", "is_opt_D", "is_opt_E",
    "prompt_overlap_count", "prompt_overlap_ratio",
    "other_overlap_mean", "other_overlap_max",
    "num_commas", "num_digits", "has_negation",
]


def build_tfidf_features(train_long, test_long, n_svd=64):
    """Fit TF-IDF on prompt+option text, add cosine sim to prompt,and return dense SVD-compressed embeddings for the option text(used as extra numeric features / as MLP input)."""
    tfidf = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), stop_words="english")
    all_text = pd.concat(
        [train_long["prompt"], train_long["option_text"],
         test_long["prompt"], test_long["option_text"]], axis=0
    ).astype(str)
    tfidf.fit(all_text)

    def sim_to_prompt(df):
        p = tfidf.transform(df["prompt"].astype(str))
        o = tfidf.transform(df["option_text"].astype(str))
        return np.array([cosine_similarity(p[i], o[i])[0, 0] for i in range(df.shape[0])])

    train_long = train_long.copy()
    test_long = test_long.copy()
    train_long["tfidf_sim_to_prompt"] = sim_to_prompt(train_long)
    test_long["tfidf_sim_to_prompt"] = sim_to_prompt(test_long)
    train_long["tfidf_sim_rank_pct"] = train_long.groupby("id")["tfidf_sim_to_prompt"].rank(pct=True)
    test_long["tfidf_sim_rank_pct"] = test_long.groupby("id")["tfidf_sim_to_prompt"].rank(pct=True)

    # SVD-compressed dense embedding of option text -> used as MLP input
    svd = TruncatedSVD(n_components=n_svd, random_state=RANDOM_STATE)
    train_opt_vecs = tfidf.transform(train_long["option_text"].astype(str))
    test_opt_vecs = tfidf.transform(test_long["option_text"].astype(str))
    svd.fit(train_opt_vecs)
    train_svd = svd.transform(train_opt_vecs)
    test_svd = svd.transform(test_opt_vecs)

    return train_long, test_long, train_svd, test_svd

## 5️⃣ Model 1 : LightGBM (Classical / Statistical Baseline)

Our first model is a **gradient-boosted decision tree (LightGBM)** trained as a binary
classifier: "does this option match the statistical/lexical profile of a correct
answer?" It consumes the engineered features above (no deep learning, no pretrained
weights) and serves as a strong, fast, highly interpretable baseline, LightGBM's
feature-importance output also tells us *which* engineered signals matter most.

Trained with **5-fold GroupKFold**, early stopping on a held-out fold, and out-of-fold
(OOF) predictions used for honest MAP@3 evaluation and for the final ensemble.

In [56]:
def run_lightgbm(train_long, test_long, groups, id_order_train, id_order_test, truth):
    import lightgbm as lgb

    feats = ENGINEERED_FEATURES + ["tfidf_sim_to_prompt", "tfidf_sim_rank_pct"]
    X, y = train_long[feats], train_long["label"]
    X_test = test_long[feats]

    gkf = GroupKFold(n_splits=N_FOLDS)
    oof = np.zeros(len(train_long))
    test_pred = np.zeros(len(test_long))

    params = dict(
        objective="binary", metric="auc", num_leaves=31, learning_rate=0.03,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=15, n_estimators=2000, random_state=RANDOM_STATE, verbosity=-1,
    )

    for tr_idx, va_idx in gkf.split(X, y, groups):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
        )
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        test_pred += model.predict_proba(X_test)[:, 1] / N_FOLDS

    evaluate("LightGBM (OOF)", train_long, id_order_train, truth, oof)
    return oof, test_pred

## 6️⃣ Model 2 : From-Scratch MLP

This satisfies the rubric's **mandatory from-scratch model** requirement. It is a
plain **feed-forward neural network implemented directly in PyTorch** : `Linear → ReLU
→ Dropout` stacked twice, ending in a single logit, with **no pretrained weights of
any kind**. Its inputs are the SVD-compressed TF-IDF embeddings plus the same
engineered statistical features used by LightGBM, standardized (zero mean, unit
variance) before training.

Each fold trains its own network with early stopping based on **validation MAP@3**
(not just loss), so the model selection criterion matches the competition metric
directly.

In [57]:
def run_mlp(train_svd, test_svd, train_extra, test_extra, train_long, test_long,
            groups, id_order_train, id_order_test, truth):
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_all = np.concatenate([train_svd, train_extra], axis=1).astype(np.float32)
    X_test_all = np.concatenate([test_svd, test_extra], axis=1).astype(np.float32)
    y_all = train_long["label"].values.astype(np.float32)

    mean, std = X_all.mean(0, keepdims=True), X_all.std(0, keepdims=True) + 1e-6
    X_all = (X_all - mean) / std
    X_test_all = (X_test_all - mean) / std

    class TabDataset(Dataset):
        def __init__(self, X, y=None):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = None if y is None else torch.tensor(y, dtype=torch.float32)

        def __len__(self):
            return len(self.X)

        def __getitem__(self, idx):
            if self.y is None:
                return self.X[idx]
            return self.X[idx], self.y[idx]

    class MLP(nn.Module):
        """Simple from-scratch feed-forward network — no pretrained weights."""
        def __init__(self, in_dim, hidden=128):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(hidden // 2, 1),
            )

        def forward(self, x):
            return self.net(x).squeeze(-1)

    gkf = GroupKFold(n_splits=N_FOLDS)
    oof = np.zeros(len(train_long))
    test_pred = np.zeros(len(test_long))

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_all, y_all, groups)):
        model = MLP(X_all.shape[1]).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        loss_fn = nn.BCEWithLogitsLoss()

        train_loader = DataLoader(
            TabDataset(X_all[tr_idx], y_all[tr_idx]), batch_size=256, shuffle=True
        )
        best_val = -1
        patience, bad_epochs = 5, 0
        best_state = None

        for epoch in range(60):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()

            model.eval()
            with torch.no_grad():
                va_logits = model(torch.tensor(X_all[va_idx]).to(device)).cpu().numpy()
            va_proba = 1 / (1 + np.exp(-va_logits))
            ranked = proba_to_ranked_letters(
                train_long.iloc[va_idx], va_proba
            )
            va_ids = train_long.iloc[va_idx]["id"].unique()
            score = map_at_3(truth.reindex(va_ids).tolist(), ranked.reindex(va_ids).tolist())
            wandb.log({"mlp/fold": fold, "mlp/epoch": epoch, "mlp/train_loss": loss.item(), "mlp/val_map3": score})

            if score > best_val:
                best_val, bad_epochs = score, 0
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            oof[va_idx] = 1 / (1 + np.exp(-model(
                torch.tensor(X_all[va_idx]).to(device)).cpu().numpy()))
            test_pred += (1 / (1 + np.exp(-model(
                torch.tensor(X_test_all).to(device)).cpu().numpy()))) / N_FOLDS
        print(f"MLP fold {fold}: best val MAP@3 = {best_val:.4f}")

    evaluate("From-scratch MLP (OOF)", train_long, id_order_train, truth, oof)
    return oof, test_pred

## 7️⃣ Models 3 & 4 : Pretrained Transformer Fine-Tuning + LoRA (Bonus)

This single reusable function implements **two rubric requirements at once** via the
`use_lora` flag:

- **Model 3 : Pretrained fine-tune:** `roberta-base` is loaded with a sequence
  classification head and **fully fine-tuned** (all weights updated) on
  `(prompt, option)` pairs, the standard Hugging Face `transformers` fine-tuning
  workflow covered in Milestone 2 ("Enter the Transformers").
- **Model 4 : Bonus / model of choice:** the same `roberta-base` backbone, but instead
  of updating all weights, we attach **LoRA adapters** (`peft` library, rank=16,
  alpha=32) to the attention query/value projection matrices and train only those
  low-rank adapters, a **parameter-efficient fine-tuning (PEFT)** approach, directly
  matching Milestone 4 ("Introduction to LoRA-finetuning, and its advantages over
  Full-finetuning").

Both variants tokenize the prompt and option as a **sentence pair** (`prompt`,
`option`), truncated/padded to 256 tokens, and are trained with `BCEWithLogitsLoss`
since we're framing this as binary "is this option correct?" classification ,  exactly
like Models 1 and 2, which keeps every model directly comparable via OOF MAP@3.

In [58]:
def run_transformer(train_long, test_long, groups, id_order_train, id_order_test,truth, model_name="roberta-base", use_lora=False, epochs=2):
    """Fine-tunes (optionally with LoRA) a pretrained transformer as a binary sequence-classification head over (prompt, option) pairs."""
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tag = f"{model_name}{' + LoRA' if use_lora else ''}"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    class PairDataset(Dataset):
        def __init__(self, df, labels=None):
            self.prompt = df["prompt"].astype(str).tolist()
            self.option = df["option_text"].astype(str).tolist()
            self.labels = labels

        def __len__(self):
            return len(self.prompt)

        def __getitem__(self, idx):
            item = {"prompt": self.prompt[idx], "option": self.option[idx]}
            if self.labels is not None:
                item["label"] = self.labels[idx]
            return item

    def collate(batch):
        prompts = [b["prompt"] for b in batch]
        options = [b["option"] for b in batch]
        enc = tokenizer(
            prompts, options, truncation=True, padding=True,
            max_length=256, return_tensors="pt",
        )
        if "label" in batch[0]:
            enc["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.float32)
        return enc

    gkf = GroupKFold(n_splits=N_FOLDS)
    y_all = train_long["label"].values.astype(np.float32)
    oof = np.zeros(len(train_long))
    test_pred = np.zeros(len(test_long))

    # For speed, transformer fine-tuning typically only needs 1 fold's worth of held-out validation on Kaggle's free GPU time budget; loop below still supports full K-fold if you have the compute/time.
    for fold, (tr_idx, va_idx) in enumerate(gkf.split(train_long, y_all, groups)):
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

        if use_lora:
            from peft import LoraConfig, get_peft_model, TaskType
            lora_cfg = LoraConfig(
                task_type=TaskType.SEQ_CLS,
                r=16, lora_alpha=32, lora_dropout=0.1,
                target_modules=["query", "value"],  # attention proj layers
            )
            model = get_peft_model(model, lora_cfg)
            model.print_trainable_parameters()

        model.to(device)

        train_loader = DataLoader(
            PairDataset(train_long.iloc[tr_idx].reset_index(drop=True), y_all[tr_idx]),
            batch_size=16, shuffle=True, collate_fn=collate,
        )
        va_loader = DataLoader(
            PairDataset(train_long.iloc[va_idx].reset_index(drop=True)),
            batch_size=32, shuffle=False, collate_fn=collate,
        )
        test_loader = DataLoader(
            PairDataset(test_long.reset_index(drop=True)),
            batch_size=32, shuffle=False, collate_fn=collate,
        )

        opt = torch.optim.AdamW(model.parameters(), lr=2e-5 if not use_lora else 1e-4)
        loss_fn = nn.BCEWithLogitsLoss()

        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                labels = batch.pop("labels")
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = labels.to(device)
                opt.zero_grad()
                logits = model(**batch).logits.squeeze(-1)
                loss = loss_fn(logits, labels)
                loss.backward()
                opt.step()
            avg_loss = loss.item()
            wandb.log({f"{tag}/fold": fold, f"{tag}/epoch": epoch, f"{tag}/train_loss": avg_loss})
            print(f"  [{model_name}{'+LoRA' if use_lora else ''}] fold {fold} epoch {epoch} done")

        def predict(loader):
            model.eval()
            preds = []
            with torch.no_grad():
                for batch in loader:
                    batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
                    logits = model(**batch).logits.squeeze(-1)
                    preds.append(torch.sigmoid(logits).cpu().numpy())
            return np.concatenate(preds)

        oof[va_idx] = predict(va_loader)
        test_pred += predict(test_loader) / N_FOLDS

        del model
        torch.cuda.empty_cache()


    evaluate(f"{tag} (OOF)", train_long, id_order_train, truth, oof)
    return oof, test_pred

## 8️⃣ Ensemble : Milestone 5 (Extracting logits, sorting, stacking predictions)

Per the Milestone 5 requirements, we:
1. **Extract and rank** the top-3 predicted labels from probabilities (already done per
   model via `proba_to_ranked_letters`)
2. **Compare models** by OOF MAP@3 and automatically select the **best 2–3 models**
3. **Stack predictions** via a probability-weighted average (weights proportional to
   each selected model's own OOF MAP@3 score), then re-rank per question to produce the
   final top-3 submission

This ensembling step is what typically pushes the final leaderboard score above what
any single model achieves alone, by combining the different biases/strengths of a
statistical model (LightGBM), a from-scratch neural net (MLP), and pretrained
transformers (RoBERTa / RoBERTa+LoRA).

In [59]:
def ensemble_and_submit(long_df, id_order, model_probas: dict, weights: dict, out_path):
    """model_probas: {name: proba_array}; weights: {name: float}"""
    combined = np.zeros(len(long_df))
    total_w = sum(weights.values())
    for name, proba in model_probas.items():
        combined += (weights[name] / total_w) * proba

    ranked = proba_to_ranked_letters(long_df, combined).reindex(id_order)
    return combined, ranked


def build_submission(id_col, ranked_letters, out_path):
    sub = pd.DataFrame({
        "ID": id_col,
        "Prediction": [" ".join(letters[:3]) for letters in ranked_letters],
    })
    sub.to_csv(out_path, index=False)
    print(f"Saved submission -> {out_path}")
    print(sub.head())
    return sub

## 9️⃣ Full Pipeline: `main()`

Ties every step above together end-to-end:
1. Load and reshape the data
2. Build engineered + TF-IDF/SVD features
3. Train Models 1–4, each reporting its own OOF MAP@3
4. Score and rank all trained models, select the top 2–3 for the ensemble
5. Build the final ensembled top-3 predictions and write `submission.csv`

The function returns each individual model's OOF score plus the final ensemble score,
these are exactly the numbers to log to **Weights & Biases** (per-model run comparison)
and to report in the **Report** deliverable's evaluation section.

In [60]:
def main(run_transformers=True, run_lora=True):
    train_wide, test_wide = load_data()
    truth = train_wide.set_index("id")["answer"]

    train_long = to_long(train_wide, is_train=True)
    test_long = to_long(test_wide, is_train=False)

    print("Adding engineered features...")
    train_long = add_engineered_features(train_long)
    test_long = add_engineered_features(test_long)

    print("Building TF-IDF / SVD features...")
    train_long, test_long, train_svd, test_svd = build_tfidf_features(train_long, test_long)

    groups = train_long["id"]
    id_order_train = train_wide["id"]
    id_order_test = test_wide["id"]

    results_oof, results_test = {}, {}


    print("\n=== Model 1: LightGBM ===")
    oof, test_pred = run_lightgbm(train_long, test_long, groups, id_order_train, id_order_test, truth)
    results_oof["lgbm"], results_test["lgbm"] = oof, test_pred


    print("\n=== Model 2: From-scratch MLP ===")
    train_extra = train_long[ENGINEERED_FEATURES + ["tfidf_sim_to_prompt", "tfidf_sim_rank_pct"]].values
    test_extra = test_long[ENGINEERED_FEATURES + ["tfidf_sim_to_prompt", "tfidf_sim_rank_pct"]].values
    oof, test_pred = run_mlp(
        train_svd, test_svd, train_extra, test_extra,
        train_long, test_long, groups, id_order_train, id_order_test, truth,
    )
    results_oof["mlp"], results_test["mlp"] = oof, test_pred


    if run_transformers:
        print("\n=== Model 3: Fine-tuned RoBERTa ===")
        oof, test_pred = run_transformer(
            train_long, test_long, groups, id_order_train, id_order_test, truth,
            model_name="roberta-base", use_lora=False, epochs=2,
        )
        results_oof["roberta"], results_test["roberta"] = oof, test_pred

        if run_lora:
            print("\n=== Model 4 (Bonus): LoRA-tuned RoBERTa ===")
            oof, test_pred = run_transformer(
                train_long, test_long, groups, id_order_train, id_order_test, truth,
                model_name="roberta-base", use_lora=True, epochs=3,
            )
            results_oof["roberta_lora"], results_test["roberta_lora"] = oof, test_pred


    print("\n=== Scoring individual models to pick ensemble members ===")
    scores = {}
    for name, oof in results_oof.items():
        ranked = proba_to_ranked_letters(train_long, oof).reindex(id_order_train)
        scores[name] = map_at_3(truth.reindex(id_order_train).tolist(), ranked.tolist())
        print(f"  {name}: {scores[name]:.4f}")

    wandb.log({"model_comparison": wandb.Table(
        data=[[name, s] for name, s in scores.items()],
        columns=["model", "map3"]
    )})
    wandb.log({"class_balance": wandb.Table(
        data=[[opt, int((truth == opt).sum())] for opt in OPTIONS],
        columns=["option", "count"]
    )})


    top_models = sorted(scores, key=scores.get, reverse=True)[:3]
    print(f"\nEnsembling top models: {top_models}")
    weights = {name: scores[name] for name in top_models}  # weight ~ OOF score

    oof_combo, _ = ensemble_and_submit(
        train_long, id_order_train,
        {n: results_oof[n] for n in top_models}, weights, None,
    )
    ensemble_score = map_at_3(
        truth.reindex(id_order_train).tolist(),
        proba_to_ranked_letters(train_long, oof_combo).reindex(id_order_train).tolist(),
    )
    print(f"Ensemble OOF MAP@3 = {ensemble_score:.4f}")
    wandb.log({"ensemble/map3": ensemble_score})
    _, test_ranked = ensemble_and_submit(
        test_long, id_order_test,
        {n: results_test[n] for n in top_models}, weights, None,
    )
    build_submission(id_order_test, test_ranked, SUBMISSION_PATH)

    return scores, ensemble_score

## 🔟 Run the Full Pipeline & Generate the Kaggle Submission

Executing `main()` trains all four models end-to-end and writes the final ensembled
predictions to `/kaggle/working/submission.csv` (or the local output path when run
outside Kaggle), ready to submit to the leaderboard.

In [61]:
if __name__ == "__main__":
    # On limited compute (e.g. CPU-only / quick local test), set
    # run_transformers=False to only run the LightGBM + MLP baseline pair.
    main(run_transformers=True, run_lora=True)
    wandb.finish()

Adding engineered features...
Building TF-IDF / SVD features...

=== Model 1: LightGBM ===
[LightGBM (OOF)] MAP@3 = 0.9928
[LightGBM (OOF)] Precision=0.9918 Recall=0.9710 F1=0.9813 Acc=0.9926

=== Model 2: From-scratch MLP ===
MLP fold 0: best val MAP@3 = 0.9932
MLP fold 1: best val MAP@3 = 0.9860
[From-scratch MLP (OOF)] MAP@3 = 0.9896
[From-scratch MLP (OOF)] Precision=0.9894 Recall=0.9330 F1=0.9604 Acc=0.9846

=== Model 3: Fine-tuned RoBERTa ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

  [roberta-base] fold 0 epoch 0 done
  [roberta-base] fold 0 epoch 1 done


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

  [roberta-base] fold 1 epoch 0 done
  [roberta-base] fold 1 epoch 1 done
[roberta-base (OOF)] MAP@3 = 0.7650
[roberta-base (OOF)] Precision=0.8487 Recall=0.1430 F1=0.2448 Acc=0.8235

=== Model 4 (Bonus): LoRA-tuned RoBERTa ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

trainable params: 1,181,185 || all params: 125,827,586 || trainable%: 0.9387
  [roberta-base+LoRA] fold 0 epoch 0 done
  [roberta-base+LoRA] fold 0 epoch 1 done
  [roberta-base+LoRA] fold 0 epoch 2 done


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

trainable params: 1,181,185 || all params: 125,827,586 || trainable%: 0.9387
  [roberta-base+LoRA] fold 1 epoch 0 done
  [roberta-base+LoRA] fold 1 epoch 1 done
  [roberta-base+LoRA] fold 1 epoch 2 done
[roberta-base + LoRA (OOF)] MAP@3 = 0.6531
[roberta-base + LoRA (OOF)] Precision=0.0000 Recall=0.0000 F1=0.0000 Acc=0.8000

=== Scoring individual models to pick ensemble members ===
  lgbm: 0.9928
  mlp: 0.9896
  roberta: 0.7650
  roberta_lora: 0.6531

Ensembling top models: ['lgbm', 'mlp', 'roberta']
Ensemble OOF MAP@3 = 0.9932
Saved submission -> /content/submission.csv
   ID Prediction
0   1      A C D
1   2      B E C
2   3      B E C
3   4      E A C
4   5      C A D


From-scratch MLP (OOF)/accuracy,▁
From-scratch MLP (OOF)/f1,▁
From-scratch MLP (OOF)/map3,▁
From-scratch MLP (OOF)/precision,▁
From-scratch MLP (OOF)/recall,▁
LightGBM (OOF)/accuracy,▁
LightGBM (OOF)/f1,▁
LightGBM (OOF)/map3,▁
LightGBM (OOF)/precision,▁
LightGBM (OOF)/recall,▁
+21,...
